In [ ]:
import os
import sys

os.chdir("..")
sys.path.append(os.getcwd())

import torch
from datasets import get_dataloaders
from models import get_model
from train import train_model
from few_shot import *
import matplotlib.pyplot as plt

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    batch_size=128,
    few_shot_k=10
)

In [ ]:
device = "cuda"

model = get_model("smallcnn", 0.3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

logs = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    epochs=20,
    device=device,
    model_path="baseline_fewshot.pth"
)

In [ ]:
plt.plot(logs["val_acc"])
plt.title("Baseline Val Accuracy")
plt.grid()
plt.show()

In [ ]:
backbone = get_model("smallcnn", 0.3)
model = ProtoNet(backbone).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_dataset = train_loader.dataset

logs = train_protonet(
    model,
    train_dataset,
    optimizer,
    device,
    epochs=20
)

In [ ]:
plt.plot(logs["loss"])
plt.title("ProtoNet Loss")
plt.grid()
plt.show()

plt.plot(logs["acc"])
plt.title("ProtoNet Accuracy")
plt.grid()
plt.show()

In [ ]:
acc = evaluate_protonet(model, train_dataset, device)
print("ProtoNet accuracy:", acc)